In [2]:
from ollama import Client
import json
import os
import re
import pandas as pd

import dataset_utils

from mvtec_dataset import MVTecDataset
from moviad.utilities.configurations import TaskType, Split, LabelName

/home/arianna_stropeni/miniconda3/envs/cbm3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
client = Client(host="http://localhost:6000")
os.environ["CUDA_VISIBLE_DEVICES"] = ""
MODEL_NAME = "gemma3:12b"
categories = ["hazelnut", "screw", "carpet"]
dataset_path = "/mnt/disk1/borsattifr/datasets/mvtec"

## Create concept list

In [ ]:
for category in categories:

    concepts = set()

    train_dataset = MVTecDataset(task = TaskType.SEGMENTATION, root = dataset_path, category = category, split = "train")
    train_dataset.load_dataset()

    test_dataset = MVTecDataset(task = TaskType.SEGMENTATION, root = dataset_path, category = category, split = "test")
    test_dataset.load_dataset()

    anomalous_samples = test_dataset.samples[test_dataset.samples.label_index == LabelName.ABNORMAL]
    print(f"Number of anomalous images: {len(anomalous_samples)}")

    normal_test_samples = test_dataset.samples[test_dataset.samples.label_index == LabelName.NORMAL]
    normal_samples = pd.concat([train_dataset.samples, normal_test_samples])
    print(f"Number of normal images: {len(normal_samples)}")

    for i in range(len(normal_samples)):
        sample = normal_samples.iloc[i]
        concept_json = dataset_utils.first_vlm_query(category, MODEL_NAME, sample, anomalous = False)
        concepts.update(c.lower() for c in concept_json)
        
    
    for i in range(len(anomalous_samples)):
        sample = anomalous_samples.iloc[i]
        concept_json = dataset_utils.first_vlm_query(category, MODEL_NAME, sample, anomalous = True)
        concepts.update(c.lower() for c in concept_json)

    concepts = list(concepts)
    
    with open(f"{category}_concepts.json", "w") as f:
        json.dump(concepts, f)


## Inspect list

In [4]:
with open("concept_lists/original/hazelnut_concepts.json", "r") as f:
    hazelnut_concepts = json.load(f)

print("Number of manually filtered concepts:", len(hazelnut_concepts))

Number of manually filtered concepts: 153


### Method one: query an LLM

In [5]:
message = "You are an industrial expert that is performing the task of visual anomaly detection."\
        f"Given the following list of visual concepts: {hazelnut_concepts}, related to images of a hazelnut, please group together those that refer to the same LITERAL meaning, i.e. if they share key words, spelling..."\
        "Ignore semantic relationships, focus only on literal wording and string similarity."\
        "Moreover, choose a representative concept that best summarizes the group."\
        "I provide two examples: 1. 'Elliptical shape': ['ellipsoidal shape', 'ellipsoid shape', 'elliptical shape']"\
        "2. 'Smooth texture': ['natural texture, 'smooth texture', 'smooth surface texture', 'smooth appearance', 'organic texture', 'uniform texture', 'consistent texture']"\
        "Avoid creating groups that are too general, e.g. grouping together all colours or all textures. Representative concepts should still be able to clearly discriminate between normal and anomalous images."\
        "Return the output as a JSON dictionary where each key is the representative concept, and its value is the list of similar concepts grouped with it."\
        "In case of concepts that cannot be grouped with any other concept, the dictionary should have both as key and value the concept itself."\
        "Please, output ONLY the JSON dictionary."

response = client.chat(model=MODEL_NAME, messages=[{"role": "user", "content": message}])

try:
    concept_json = dataset_utils.extract_json(response["message"]["content"])
except Exception as e:
    print(f"Error parsing response: {e}")
    concept_json = []

In [ ]:
print(concept_json)

In [9]:
print("Number of concepts extracted by the LLM:", len(concept_json))
print("Extracted concepts:", concept_json.keys())

Number of concepts extracted by the LLM: 29
Extracted concepts: dict_keys(['Crack', 'Hole', 'Split', 'Structural Damage', 'Shell Fracture', 'Shape Distortion', 'Surface Imperfection', 'Color Variation', 'Print Artifacts', 'Edge Blurring', 'Texture', 'Compact Appearance', 'Blurred Details', 'Visible Seam', 'Dark Interior', 'Light Streaks', 'Sharp Line', 'Organic Appearance', 'Cut Surface', 'Consistent Appearance', 'Smudged Texture', 'Discontinuity', 'Missing Material', 'Pattern Distortion', 'Uniform Surface', 'Reduced Contrast', 'White Spots', 'Consistent Color', 'Surface Disruption'])


### Method two: cluster embeddings

In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(hazelnut_concepts)

clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=0.5, linkage="average")
labels = clustering.fit_predict(embeddings)

In [22]:
from collections import defaultdict

grouped = defaultdict(list)
for concept, label in zip(hazelnut_concepts, labels):
    grouped[label].append(concept)

grouped_concepts = list(grouped.values())

print("Number of concepts extracted from clustering:", len(grouped_concepts))
print(grouped_concepts)

Number of concepts extracted from clustering: 134
[['visible gap'], ['structural damage'], ['blurry area'], ['consistent appearance'], ['white spots'], ['round shape', 'rounded shape'], ['surface crack'], ['visible hole'], ['brown color', 'brown color'], ['rough surface'], ['oval shape', 'oval shape'], ['smudged texture'], ['textured surface'], ['uniform texture'], ['blurred edges', 'blurry edges', 'edge blurring', 'blurred edge'], ['structural breach'], ['texture variation'], ['smooth surface'], ['natural product'], ['compact structure'], ['cut surface'], ['natural form'], ['defined texture'], ['crack', 'crack'], ['shell fracture', 'fractured shell'], ['organic appearance'], ['visible cracks', 'visible crack'], ['structural break'], ['visible cut'], ['surface discontinuity'], ['visible ridges'], ['textured appearance'], ['color variation'], ['visible ink'], ['consistent surface'], ['compact form'], ['uneven split'], ['defined ridges'], ['color contrast'], ['shape distortion'], ['fract